# COSMIC Data Engineer : Interview tasks

## 1. BASH

### Question 1.1: How to properly track errors. 

In Bash, we can use the "exit {non-zero}" to quit the run, following an error and "exit 0" to quit after a success. In order to track them, returning the result of command with a proper indication to a file is an easy solution. For example:

In [ ]:
if [[ $? -ne 0 ]]; then
    echo "ERROR: cp failed" >&2
    exit 1
fi 

After using my own knowledge and conducting further research, I found that we could simply use the rafter, like when we write into a file. It would redirect the stderr (2) to a log file next to the stdout (1). Here the 2 represents a standard error and 1 represents the standard output. For example:

In [ ]:
mycommand >> output_command.log 2 >> error_comand.log 

In Python, the use of logging is recommended. Using logging.error will allow us, in case of error, to flag them in the logging file.

In [ ]:
import logging

logging.error(f'the file is not opened')

With that said, we can still return an error to bash while executing the Python script from there. With a try expect scenario, the Python script will return 0 on success and 1 in exceptions. We can then catch these error (or success) codes in bash. For example:

In [ ]:
import sys 

try:
    #execute Code
except Exception as e:
    print(f'error : {e}')
    sys.exit(1)

### Question 1.2: If your Bash script contains asynchronized commands, such as IBM LSF or Slurm job schedulers, how to compose your script to make sure everything works well and exits with a proper code?

Asynchronized commands will return a code when the command has started, but an asynchron job can still fail. In order to catch the error, we need to wait for the end of the execution to get the result. These jobs have an id We can collect this id and check the status of the job regularly throughout the script. If we are working on a pipeline, we can establish rules that will make the next step start only when the task has been successful.

### Question 1.3: Why the code comment is important in Bash? And how to give proper comments in a Bash script?

Code comment is important in every language, but bash can become unclear using various commands piped one to others. Keeping a clear and explicit commentary will help anyone working or executing the bash script. It is common to use a part of a code found online that works. However, where we are not explicitly aware of every detail, keeping comments can help to efficiently get back to it later on.

To write efficient comments, we first need a good header, explaining the purpose of the script, the date, the editor and their information if we need to contact them. We can also specify how to use the code with the different options that have been implemented. I believe that a clear descriptor of what and why is needed before every step. If some parts are less clear, we can comment specific commands. When we stop editing a script, it is important to note what future implementations/improvements are needed.

## 2. Data processing

### Question 2.1

For this question, we will first inspect the data. From this, it will be easier to apply filters and build the final result. In a separate file, COSMIC/question2_analysis.py, we can find the function, and in COSMIC/test_question2.py, we can find the tests.

In [ ]:
import pandas as pd
filepath = "COSMIC/simple_somatic_mutation.open.BLCA-CN.tsv.gz"
df = pd.read_csv(filepath, sep="\t", compression="gzip", low_memory=False)
df.head()

,icgc_mutation_id,icgc_donor_id,project_code,icgc_specimen_id,icgc_sample_id,matched_icgc_sample_id,submitted_sample_id,submitted_matched_sample_id,chromosome,chromosome_start,...,experimental_protocol,sequencing_strategy,base_calling_algorithm,alignment_algorithm,variation_calling_algorithm,other_analysis_algorithm,seq_coverage,raw_data_repository,raw_data_accession,initial_data_release_date
0,MU5219,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,3,178936091,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN
1,MU5219,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,3,178936091,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN
2,MU4559679,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,12,56558254,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN
3,MU4559679,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,12,56558254,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN
4,MU4559679,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,12,56558254,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN


The data has been loaded. We can look at the first results and go through the various columns. In our first 5 results, we already have duplicates which could affect our count. Therefore, we must drop the duplicates. 

In [49]:
unique_mutations = df.drop_duplicates(subset=["icgc_mutation_id"])

In [50]:
unique_mutations.head()

,icgc_mutation_id,icgc_donor_id,project_code,icgc_specimen_id,icgc_sample_id,matched_icgc_sample_id,submitted_sample_id,submitted_matched_sample_id,chromosome,chromosome_start,...,experimental_protocol,sequencing_strategy,base_calling_algorithm,alignment_algorithm,variation_calling_algorithm,other_analysis_algorithm,seq_coverage,raw_data_repository,raw_data_accession,initial_data_release_date
0,MU5219,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,3,178936091,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN
2,MU4559679,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,12,56558254,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN
30,MU4626274,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,X,48047128,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN
34,MU3888756,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,17,39253835,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN
36,MU4603099,DO48399,BLCA-CN,SP106410,SA514847,SA514849,B54-Tumor,B54-Blood,1,171251414,...,Agilent SureSelect in Solution http://www.halo...,WXS,Illumina base-calling pipeline http://www.illu...,BWA http://bio-bwa.sourceforge.net/bwa.shtml,VarScan http://varscan.sourceforge.net/somatic...,Annotation with annovar http://www.openbioinfo...,70.0,EGA,EGAS00001000677,NaN


We will group the mutated from allele and mutated to allele, in front of icgc mutation idc. This should give us the number of unique mutations associated to every possible pattern.

In [51]:
result = (
    unique_mutations
    .groupby(["mutated_from_allele", "mutated_to_allele"])["icgc_mutation_id"]
)

In [53]:
result_count = result.count()
result_count

mutated_from_allele  mutated_to_allele
A                    C                     446
                     G                     923
                     T                     467
C                    A                     811
                     G                    1404
                     T                    3805
G                    A                    3620
                     C                    1376
                     T                     660
T                    A                     460
                     C                     923
                     G                     583
Name: icgc_mutation_id, dtype: int64

We have obtained the right result, but we want a different visualisation. We can reset the index and sort the values to break down the grouped columns.

In [54]:
result = (
    result_count
    .reset_index()
    .sort_values(["mutated_from_allele", "mutated_to_allele"])
)

In [55]:
result

,mutated_from_allele,mutated_to_allele,icgc_mutation_id
0,A,C,446
1,A,G,923
2,A,T,467
3,C,A,811
4,C,G,1404
5,C,T,3805
6,G,A,3620
7,G,C,1376
8,G,T,660
9,T,A,460


### Question 2.2: Please find out which icgc_sample_id has the highest and lowest unique icgc_mutation_id count.

For this question the logic is similar. We deduplicate one row per (sample, mutation) pair, and then by grouping and counting, we can find the extrema.

In [69]:
unique_per_sample = df.drop_duplicates(subset=["icgc_sample_id", "icgc_mutation_id"])

counts = (
    unique_per_sample
    .groupby("icgc_sample_id")["icgc_mutation_id"]
    .count()
    .reset_index()
)

In [72]:
highest = counts.loc[counts["icgc_mutation_id"].idxmax()]
lowest  = counts.loc[counts["icgc_mutation_id"].idxmin()]

extremes = {
    "highest": {"sample_id": highest["icgc_sample_id"],
                "count":     int(highest["icgc_mutation_id"])},
    "lowest":  {"sample_id": lowest["icgc_sample_id"],
                "count":     int(lowest["icgc_mutation_id"])},
}

print(f"Highest unique mutations -> sample: {extremes['highest']['sample_id']}  "
        f"count: {extremes['highest']['count']}")
print(f"Lowest  unique mutations -> sample: {extremes['lowest']['sample_id']}  "
        f"count: {extremes['lowest']['count']}")

Highest unique mutations -> sample: SA514800  count: 583
Lowest  unique mutations -> sample: SA514876  count: 14


## 3. Database

### Question 3.1: How many genes in the gene table have an id_biotype of 23?

In [ ]:
SELECT count(ID_GENE) as count_biotype_23
FROM gene
where ID_BIOTYPE = 23

174

### Question 3.2: What is the Ensembl Gene ID for the Gene_symbol TTTY2?

In [ ]:
Select ENSEMBL_GENE_ID 
From gene
Where GENE_SYMBOL Like N'TTTY2'

ENSG00000212855

### Question 3.3: Give a breakdown of the number of genes for each chromosome.

In [ ]:
Select count(ID_GENE) as count_gene_per_chr, CHROMOSOME 
From gene 
GROUP BY CHROMOSOME;

|count_gene_per_chr	|CHROMOSOME|
| ----------------- | -------- |
| 51	| 1 |
| 25	| 2 |
| 31	| 3 |
| 20	| 4 |
| 25	| 5 |
| 16	| 6 |
| 19	| 7 |
| 25	| 8 |
| 18	| 9 |
| 21	| 10 |
| 25	| 11 |
| 21	| 12 |
| 16	| 13 |
| 18	| 14 |
| 17	| 15 |
| 28	| 16 |
| 27	| 17 |
| 9	| 18 |
| 27	| 19 |
| 16	| 20 |
| 9	| 21 |
| 16	| 22 |
| 16	| 23 |
| 4	| 24 |

### Question 3.4: How many Transcripts does the Gene Symbol ﻿ RAI14 has?

In [ ]:
Select count(ID_TRANSCRIPT) as count_Transcripts_RAI14
From transcript as t join gene as g on g.ID_GENE = t.ID_GENE
Where Gene_symbol Like N'RAI14';

29

### Question 3.5: What is the canonical transcript accession for Ensembl Gene id ﻿ ENSG00000266960?

In [ ]:
Select ACCESSION
From transcript as t 
join gene as g on g.ID_GENE = t.ID_GENE
Where ENSEMBL_GENE_ID = N'ENSG00000266960' 
and IS_CANONICAL = 'y';

ENST00000586416

### Question 3.6:List the Transcript accessions for the Gene Symbol ﻿ AK1 with id_biotype 23 and flags gencode_basic

In [ ]:
Select ACCESSION
From transcript as t join gene as g on g.ID_GENE = t.ID_GENE 
where GENE_SYMBOL = 'AK1' and g.ID_BIOTYPE = 23 and FLAGS = 'gencode_basic’;

ENST00000223836
ENST00000373156
ENST00000373176

### Question 3.7: Imagine that we have a table called “some_gene” with only a subset of the gene data. If I want to join the gene table with this table but display all the genes in the result, what kind of join would you do?

In [ ]:
Left with gene table on the left 
Select g.*, sg.some_column 
From gene as g left join some_gene as sg on sg.ID_GENE = g.ID_GENE;

### Question 3.8: Imagine that the gene and transcript tables are getting very big and that joining the two tables get slower and slower. What would you do to improve performances?

I would start by adding indexes on join keys. I would focus on the columns mainly used, and create indexes. We can partition the database in smaller subsets if the queries allow it. Some large databases like Mgnify for example, made the choice to store their data in formats like parquet allowing larger datasets. A view could also be used.

### Question 3.9: If you want to avoid duplicates in a table, what kind of index would you create?

A unique index can be created, or a unique constraint.

### Question 3.10: If you want to make sure that all the id_gene ids in the transcript table exists in the gene table, what kind of index would you create?

A foreign key needs to be set between the two tables.

### Feedback : 

The tasks review a large panel of common use case in a day to day. The questions are clear and well made.  